# AI Programming — Lecture 20
## Mini I-JEPA: Self-Supervised Representation Learning

이번 실습에서는 **I-JEPA의 핵심 아이디어**를 작은 CIFAR-10 예제로 확인합니다.

공식 I-JEPA 전체를 재현하는 것이 아니라, 학부 실습에서 구조를 이해하기 위한 **Mini I-JEPA**입니다.

### 핵심 아이디어

```text
visible context
    ↓
Context Encoder
    ↓
Predictor ───────────────→ target representation
                              ↑
full image → Target Encoder ─┘
```

- Self-supervised pretraining에서는 CIFAR-10 label을 사용하지 않습니다.
- Context encoder는 visible patch만 입력받습니다.
- Target encoder에는 **전체 이미지**가 들어갑니다.
- Target encoder 출력에서 target 위치 representation만 선택합니다.
- Predictor는 target pixel을 보지 않고 **target 위치 query**만 받습니다.
- Target encoder는 backpropagation이 아니라 **EMA**로 업데이트합니다.
- 마지막에는 **Linear Probe**로 representation을 평가합니다.

> MAE가 masked pixel을 복원한다면, JEPA는 masked region의 **latent representation**을 예측합니다.

## 0. 실습 환경 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

IMAGE_SIZE = 32
PATCH_SIZE = 4
GRID_SIZE = IMAGE_SIZE // PATCH_SIZE
NUM_PATCHES = GRID_SIZE * GRID_SIZE
PATCH_DIM = PATCH_SIZE * PATCH_SIZE * 3

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
ENCODER_DEPTH = 2

TARGET_BLOCK = 3
EMA_MOMENTUM = 0.99

SSL_TRAIN_SAMPLES = 10000
SSL_BATCH_SIZE = 128
SSL_EPOCHS = 5
SSL_LR = 1e-3

PROBE_TRAIN_SAMPLES = 10000
PROBE_TEST_SAMPLES = 5000
PROBE_EPOCHS = 10

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "CPU")
print("Patch grid:", GRID_SIZE, "x", GRID_SIZE)
print("Number of patches:", NUM_PATCHES)

## 1. CIFAR-10 불러오기

CIFAR-10에는 label이 있지만 **JEPA pretraining 단계에서는 사용하지 않습니다**.
Label은 마지막 Linear Probe에서만 사용합니다.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

## 2. Image를 Patch로 나누기

$32 \times 32$ 이미지를 $4 \times 4$ patch로 나누면
$8 \times 8 = 64$개의 patch token이 만들어집니다.

In [ ]:
def patchify(images):
    patches = tf.image.extract_patches(
        images=images,
        sizes=[1, PATCH_SIZE, PATCH_SIZE, 1],
        strides=[1, PATCH_SIZE, PATCH_SIZE, 1],
        rates=[1, 1, 1, 1],
        padding="VALID",
    )
    b = tf.shape(images)[0]
    return tf.reshape(patches, [b, NUM_PATCHES, PATCH_DIM])

sample_patches = patchify(tf.convert_to_tensor(x_train[:4]))
print("Patch tensor:", sample_patches.shape)

## 3. Context와 Target Block 만들기

매 training step마다 $8 \times 8$ patch grid에서 $3 \times 3$ target block 하나를 선택합니다.

```text
64 patches
├─ target : 9 patches
└─ context: 55 patches
```

**중요:** Target encoder에 target patch만 넣는 것이 아닙니다.  
Target encoder는 전체 64 patch를 모두 처리한 뒤, 출력에서 target 위치만 선택합니다.

In [ ]:
def sample_block_indices(grid_size=GRID_SIZE, block_size=TARGET_BLOCK):
    max_start = grid_size - block_size
    row = np.random.randint(0, max_start + 1)
    col = np.random.randint(0, max_start + 1)

    target = []
    for r in range(row, row + block_size):
        for c in range(col, col + block_size):
            target.append(r * grid_size + c)

    target = np.array(target, dtype=np.int32)
    target_set = set(target.tolist())

    context = np.array(
        [i for i in range(grid_size * grid_size) if i not in target_set],
        dtype=np.int32,
    )
    return context, target

context_idx, target_idx = sample_block_indices()
print("Context:", len(context_idx))
print("Target :", len(target_idx))
print("Target indices:", target_idx)

### Target block 시각화

아래 검은 영역은 설명을 위한 시각화입니다.  
실제 context encoder에는 검은 pixel을 넣지 않고 **해당 patch token을 제거**합니다.

In [ ]:
def visualize_mask(image, target_indices):
    masked = image.copy()

    for idx in target_indices:
        r, c = idx // GRID_SIZE, idx % GRID_SIZE
        r0, r1 = r * PATCH_SIZE, (r + 1) * PATCH_SIZE
        c0, c1 = c * PATCH_SIZE, (c + 1) * PATCH_SIZE
        masked[r0:r1, c0:c1] = 0.0

    plt.figure(figsize=(8, 3))

    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(masked)
    plt.title("Context encoder")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(image)
    plt.title("Target encoder: full image")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

_, target_example = sample_block_indices()
visualize_mask(x_train[0], target_example)

## 4. Tiny Transformer Block

In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x, training=False):
        a = self.attn(x, x, training=training)
        x = self.norm1(x + a)

        f = self.ffn(x, training=training)
        return self.norm2(x + f)

## 5. Context / Target Encoder

두 encoder는 같은 구조를 사용합니다.

```text
Patch
→ Dense projection
→ Position embedding
→ Transformer blocks
→ Latent representation
```

Context encoder는 선택된 visible patch만 처리하고,
Target encoder는 전체 patch를 처리합니다.

In [ ]:
class PatchTransformerEncoder(keras.Model):
    def __init__(self, num_patches, embed_dim, num_heads, ff_dim, depth):
        super().__init__()

        self.projection = layers.Dense(embed_dim)

        self.position_embedding = self.add_weight(
            name="position_embedding",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal",
            trainable=True,
        )

        self.blocks = [
            TransformerBlock(embed_dim, num_heads, ff_dim)
            for _ in range(depth)
        ]

        self.final_norm = layers.LayerNormalization()

    def call(self, patches, indices, training=False):
        selected = tf.gather(patches, indices, axis=1)
        x = self.projection(selected)

        pos = tf.gather(
            self.position_embedding[0],
            indices,
            axis=0,
        )

        x = x + pos[None, :, :]

        for block in self.blocks:
            x = block(x, training=training)

        return self.final_norm(x)

## 6. Predictor

Predictor는 target image content를 보지 않습니다.

```text
Q   = target 위치 query
K,V = context representations
```

따라서 predictor는 **어디를 예측해야 하는지**는 알지만,
그 위치의 실제 pixel은 알지 못합니다.

In [ ]:
class JEPAPredictor(keras.Model):
    def __init__(self, num_patches, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.target_token = self.add_weight(
            name="target_query_token",
            shape=(1, 1, embed_dim),
            initializer="random_normal",
            trainable=True,
        )

        self.target_position_embedding = self.add_weight(
            name="target_position_embedding",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal",
            trainable=True,
        )

        self.cross_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
        )

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(embed_dim),
        ])

    def call(self, context_repr, target_indices, training=False):
        batch_size = tf.shape(context_repr)[0]

        target_pos = tf.gather(
            self.target_position_embedding[0],
            target_indices,
            axis=0,
        )

        query = self.target_token + target_pos[None, :, :]
        query = tf.repeat(query, repeats=batch_size, axis=0)

        predicted = self.cross_attention(
            query=query,
            key=context_repr,
            value=context_repr,
            training=training,
        )

        x = self.norm1(query + predicted)
        return self.norm2(x + self.ffn(x, training=training))

## 7. Online / Target Network 초기화

In [ ]:
online_encoder = PatchTransformerEncoder(
    NUM_PATCHES, EMBED_DIM, NUM_HEADS, FF_DIM, ENCODER_DEPTH
)

target_encoder = PatchTransformerEncoder(
    NUM_PATCHES, EMBED_DIM, NUM_HEADS, FF_DIM, ENCODER_DEPTH
)

predictor = JEPAPredictor(
    NUM_PATCHES, EMBED_DIM, NUM_HEADS, FF_DIM
)

all_indices = np.arange(NUM_PATCHES, dtype=np.int32)

dummy_images = tf.zeros((2, IMAGE_SIZE, IMAGE_SIZE, 3))
dummy_patches = patchify(dummy_images)
dummy_context, dummy_target = sample_block_indices()

context_repr = online_encoder(dummy_patches, dummy_context)
_ = target_encoder(dummy_patches, all_indices)
_ = predictor(context_repr, dummy_target)

target_encoder.set_weights(online_encoder.get_weights())
target_encoder.trainable = False

print("Online encoder params:", online_encoder.count_params())
print("Predictor params:", predictor.count_params())

## 8. Latent Prediction Loss

이번 교육용 구현에서는 predicted representation과 target representation을
L2-normalize한 뒤 MSE를 사용합니다.

$$
\mathcal{L}
=
\left\|
\hat{\mathbf z}
-
\mathrm{sg}(\mathbf z)
\right\|_2^2
$$

`sg`는 stop-gradient입니다.

In [ ]:
def compute_jepa_loss(images, context_indices, target_indices, training=True):
    patches = patchify(images)

    # Context encoder: visible patches only
    context_repr = online_encoder(
        patches,
        context_indices,
        training=training,
    )

    # Target encoder: FULL image
    target_full = target_encoder(
        patches,
        all_indices,
        training=False,
    )

    # Target 위치 representation만 선택
    target_repr = tf.gather(
        target_full,
        target_indices,
        axis=1,
    )

    target_repr = tf.stop_gradient(target_repr)

    # Predictor: context + target position query
    predicted_repr = predictor(
        context_repr,
        target_indices,
        training=training,
    )

    predicted_norm = tf.math.l2_normalize(predicted_repr, axis=-1)
    target_norm = tf.math.l2_normalize(target_repr, axis=-1)

    loss = tf.reduce_mean(
        tf.square(predicted_norm - target_norm)
    )

    cosine = tf.reduce_mean(
        tf.reduce_sum(predicted_norm * target_norm, axis=-1)
    )

    return loss, cosine

## 9. EMA Target Encoder

Target encoder는 optimizer로 학습하지 않고

$$
\theta_{target}
\leftarrow
m\theta_{target}
+
(1-m)\theta_{online}
$$

으로 업데이트합니다.

In [ ]:
def update_target_encoder(
    online_model,
    target_model,
    momentum=EMA_MOMENTUM,
):
    for online_w, target_w in zip(
        online_model.weights,
        target_model.weights,
    ):
        target_w.assign(
            momentum * target_w
            + (1.0 - momentum) * online_w
        )

## 10. Self-Supervised Dataset

Pretraining dataset에는 **image만** 넣습니다. Label은 사용하지 않습니다.

In [ ]:
x_ssl = x_train[:SSL_TRAIN_SAMPLES]

ssl_dataset = (
    tf.data.Dataset
    .from_tensor_slices(x_ssl)
    .shuffle(SSL_TRAIN_SAMPLES, seed=SEED)
    .batch(SSL_BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("SSL images:", x_ssl.shape)

## 11. Mini I-JEPA Pretraining

각 batch에서:

```text
Random target block
→ Context encoder
→ Target encoder (full image)
→ Predictor
→ Latent loss
→ Backpropagation: online encoder + predictor
→ EMA: target encoder
```

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=SSL_LR)

ssl_loss_history = []
ssl_cosine_history = []

for epoch in range(SSL_EPOCHS):
    losses = []
    cosines = []

    for images in ssl_dataset:
        context_indices, target_indices = sample_block_indices()

        with tf.GradientTape() as tape:
            loss, cosine = compute_jepa_loss(
                images,
                context_indices,
                target_indices,
                training=True,
            )

        variables = (
            online_encoder.trainable_variables
            + predictor.trainable_variables
        )

        gradients = tape.gradient(loss, variables)
        optimizer.apply_gradients(zip(gradients, variables))

        update_target_encoder(
            online_encoder,
            target_encoder,
        )

        losses.append(float(loss))
        cosines.append(float(cosine))

    mean_loss = np.mean(losses)
    mean_cosine = np.mean(cosines)

    ssl_loss_history.append(mean_loss)
    ssl_cosine_history.append(mean_cosine)

    print(
        f"Epoch {epoch+1:02d}/{SSL_EPOCHS} | "
        f"loss={mean_loss:.5f} | "
        f"cosine={mean_cosine:.4f}"
    )

## 12. Pretraining 결과 확인

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(ssl_loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Latent Prediction Loss")
plt.title("Mini I-JEPA Pretraining Loss")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(ssl_cosine_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cosine Similarity")
plt.title("Predicted vs. Target Representation")
plt.grid(alpha=0.3)
plt.show()

# Part II. Linear Probe

Self-supervised loss가 감소했다고 해서 좋은 representation이 보장되는 것은 아닙니다.

Encoder를 고정하고 linear classifier만 학습하여 representation 품질을 확인합니다.

```text
Frozen Encoder
→ Global Average Pooling
→ Linear Classifier
→ CIFAR-10 label
```

비교:

```text
Random Encoder
vs.
Mini I-JEPA Encoder
```

> 짧은 교육용 pretraining이므로 JEPA가 항상 큰 성능 차이로 이기지는 않을 수 있습니다.

## 13. Encoder Feature 추출

In [ ]:
def extract_features(encoder, images, batch_size=256):
    dataset = tf.data.Dataset.from_tensor_slices(images).batch(batch_size)
    features = []

    for batch in dataset:
        patches = patchify(batch)

        reps = encoder(
            patches,
            all_indices,
            training=False,
        )

        # 전체 patch representation 평균
        image_features = tf.reduce_mean(reps, axis=1)
        features.append(image_features.numpy())

    return np.concatenate(features, axis=0)

In [ ]:
probe_x_train = x_train[:PROBE_TRAIN_SAMPLES]
probe_y_train = y_train[:PROBE_TRAIN_SAMPLES]

probe_x_test = x_test[:PROBE_TEST_SAMPLES]
probe_y_test = y_test[:PROBE_TEST_SAMPLES]

jepa_train_features = extract_features(
    online_encoder,
    probe_x_train,
)

jepa_test_features = extract_features(
    online_encoder,
    probe_x_test,
)

print(jepa_train_features.shape)
print(jepa_test_features.shape)

## 14. Random Encoder Baseline

In [ ]:
random_encoder = PatchTransformerEncoder(
    NUM_PATCHES, EMBED_DIM, NUM_HEADS, FF_DIM, ENCODER_DEPTH
)

_ = random_encoder(dummy_patches, all_indices)

random_train_features = extract_features(
    random_encoder,
    probe_x_train,
)

random_test_features = extract_features(
    random_encoder,
    probe_x_test,
)

print(random_train_features.shape)

## 15. Linear Classifier 학습

In [ ]:
def train_linear_probe(
    train_features,
    train_labels,
    test_features,
    test_labels,
):
    classifier = keras.Sequential([
        layers.Input(shape=(train_features.shape[1],)),
        layers.Dense(10),
    ])

    classifier.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-2),
        loss=keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
        metrics=["accuracy"],
    )

    history = classifier.fit(
        train_features,
        train_labels,
        validation_split=0.2,
        epochs=PROBE_EPOCHS,
        batch_size=128,
        verbose=0,
    )

    _, test_acc = classifier.evaluate(
        test_features,
        test_labels,
        verbose=0,
    )

    return history, test_acc

## 16. Random vs. Mini I-JEPA 비교

In [ ]:
random_history, random_acc = train_linear_probe(
    random_train_features,
    probe_y_train,
    random_test_features,
    probe_y_test,
)

jepa_history, jepa_acc = train_linear_probe(
    jepa_train_features,
    probe_y_train,
    jepa_test_features,
    probe_y_test,
)

print(f"Random Encoder Linear Probe: {random_acc:.4f}")
print(f"Mini I-JEPA Linear Probe   : {jepa_acc:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(
    random_history.history["val_accuracy"],
    label="Random Encoder",
)
plt.plot(
    jepa_history.history["val_accuracy"],
    label="Mini I-JEPA Encoder",
)
plt.xlabel("Linear Probe Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Representation Quality")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 17. 직접 해보기

1. `TARGET_BLOCK = 2`와 `3`을 비교하세요.
2. `SSL_EPOCHS = 3`과 `10`을 비교하세요.
3. `EMA_MOMENTUM = 0.9`, `0.99`를 비교하세요.
4. Target encoder가 전체 64 patch를 입력받는 부분을 찾으세요.
5. Predictor의 `Q`, `K`, `V`가 각각 무엇인지 설명하세요.
6. Random encoder와 Mini I-JEPA encoder의 Linear Probe 결과를 비교하세요.
7. Target encoder에 target patch만 넣으면 현재 방식과 무엇이 달라질지 생각해 보세요.

# 18. 정리

### MAE

```text
Visible patches
→ Encoder
→ Decoder
→ Missing pixels
```

### I-JEPA

```text
Visible context
→ Context Encoder
→ Predictor
→ Target latent representation
         ↑
 Full image
 → EMA Target Encoder
```

### 꼭 기억할 것

1. **JEPA는 pixel이 아니라 latent representation을 예측합니다.**
2. **Context encoder는 visible patch만 봅니다.**
3. **Target encoder에는 전체 이미지가 들어갑니다.**
4. **Target representation은 target encoder 출력에서 선택합니다.**
5. **Predictor는 target content가 아니라 target 위치 query를 사용합니다.**
6. **Target encoder는 EMA로 업데이트합니다.**
7. **Representation 품질은 Linear Probe와 같은 downstream evaluation으로 확인할 수 있습니다.**

이 notebook은 개념 학습을 위한 Mini I-JEPA이며,
공식 대규모 I-JEPA의 masking strategy와 optimization을 모두 재현하지는 않습니다.